In [1]:
import numpy as np, torch, torchvision, pandas as pd
print("numpy:", np.__version__, "| torch:", torch.__version__, "| torchvision:", torchvision.__version__)

import os
base = "/kaggle/input"
for root, dirs, files in os.walk(base):
    depth = root[len(base):].count(os.sep)
    if depth <= 3:
        print(root, "-> dirs:", dirs[:5], f"| files: {len(files)}")
    if depth > 3:
        break

numpy: 2.0.2 | torch: 2.10.0+cu128 | torchvision: 0.25.0+cu128
/kaggle/input -> dirs: ['notebooks'] | files: 0
/kaggle/input/notebooks -> dirs: ['vidhushinikg'] | files: 0
/kaggle/input/notebooks/vidhushinikg -> dirs: ['age-prediction-preprocessing'] | files: 0
/kaggle/input/notebooks/vidhushinikg/age-prediction-preprocessing -> dirs: ['__results___files', 'processed_faces_10_80'] | files: 5


In [2]:
DATA_DIR = "/kaggle/input/notebooks/vidhushinikg/age-prediction-preprocessing/processed_faces_10_80"

import os
print(os.listdir(DATA_DIR)[:10])  # should show train.csv, val.csv, test.csv, and .jpg files

import pandas as pd
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(train_df.head())

['2936_15.jpg', '21957_47.jpg', '11682_29.jpg', '29905_60.jpg', '36289_70.jpg', '40482_80.jpg', '9019_25.jpg', '37958_73.jpg', '31286_62.jpg', '29847_60.jpg']
Train: 28107 | Val: 6023 | Test: 6023
       filename  age
0   8754_25.jpg   25
1   3345_16.jpg   16
2  15290_35.jpg   35
3  12625_31.jpg   31
4    944_12.jpg   12


In [3]:
!pip install -q --no-deps timm
import timm
print("timm imported OK, version:", timm.__version__)

timm imported OK, version: 1.0.26


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

MIN_AGE, MAX_AGE = 10, 80
NUM_CLASSES = MAX_AGE - MIN_AGE + 1  # 71 age classes

class HybridAgeModel(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, freeze_backbones=True):
        super().__init__()
        self.resnet = timm.create_model('resnet50', pretrained=True, num_classes=0)
        self.swin = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=0)

        if freeze_backbones:
            for p in self.resnet.parameters(): p.requires_grad = False
            for p in self.swin.parameters(): p.requires_grad = False
            for p in self.resnet.layer4.parameters(): p.requires_grad = True
            for p in self.swin.layers[-1].parameters(): p.requires_grad = True

        fusion_dim = 2048 + 768
        self.fusion = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
        )
        self.head = nn.Linear(512, num_classes)

    def forward(self, x):
        feat_a = self.resnet(x)
        feat_b = self.swin(x)
        fused = torch.cat([feat_a, feat_b], dim=1)
        fused = self.fusion(fused)
        return self.head(fused)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HybridAgeModel().to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Device: {device}")
print(f"Trainable: {trainable:,} / Total: {total:,}")

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Device: cuda
Trainable: 31,810,039 / Total: 52,506,113


In [5]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

class FaceAgeDataset(Dataset):
    def __init__(self, csv_path, img_dir, train=True):
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        if train:
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.RandomHorizontalFlip(0.5),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
                transforms.RandomRotation(10),
                transforms.ToTensor(),
                transforms.Normalize(MEAN, STD),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(MEAN, STD),
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row["filename"])).convert("RGB")
        img = self.transform(img)
        age = int(row["age"]) - MIN_AGE
        return img, age

train_ds = FaceAgeDataset(os.path.join(DATA_DIR, "train.csv"), DATA_DIR, train=True)
val_ds   = FaceAgeDataset(os.path.join(DATA_DIR, "val.csv"),   DATA_DIR, train=False)
test_ds  = FaceAgeDataset(os.path.join(DATA_DIR, "test.csv"),  DATA_DIR, train=False)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")


Train: 28107 | Val: 6023 | Test: 6023


In [6]:
from tqdm import tqdm

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))

def make_gaussian_label_distribution(true_classes, num_classes=NUM_CLASSES, sigma=2.0, device="cpu"):
    classes = torch.arange(num_classes, device=device).float().unsqueeze(0)
    true_classes = true_classes.float().unsqueeze(1).to(device)
    dist = torch.exp(-0.5 * ((classes - true_classes) / sigma) ** 2)
    return dist / dist.sum(dim=1, keepdim=True)

def kl_div_loss(logits, target_dist):
    log_probs = F.log_softmax(logits, dim=1)
    return F.kl_div(log_probs, target_dist, reduction="batchmean")

def expected_age_class(logits):
    probs = F.softmax(logits, dim=1)
    classes = torch.arange(NUM_CLASSES, device=logits.device).float()
    return (probs * classes).sum(dim=1)

print("Training components defined.")

Training components defined.


/tmp/ipykernel_23/3872954479.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))


In [7]:
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    train_loss, train_mae = 0.0, 0.0

    for imgs, ages in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        imgs, ages = imgs.to(device), ages.to(device)
        target_dist = make_gaussian_label_distribution(ages, device=device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=(device == "cuda")):
            logits = model(imgs)
            loss = kl_div_loss(logits, target_dist)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * imgs.size(0)
        preds = expected_age_class(logits)
        train_mae += torch.abs(preds - ages.float()).sum().item()

    train_loss /= len(train_ds)
    train_mae /= len(train_ds)

    model.eval()
    val_mae = 0.0
    with torch.no_grad():
        for imgs, ages in val_loader:
            imgs, ages = imgs.to(device), ages.to(device)
            logits = model(imgs)
            preds = expected_age_class(logits)
            val_mae += torch.abs(preds - ages.float()).sum().item()
    val_mae /= len(val_ds)

    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} train_MAE={train_mae:.2f} val_MAE={val_mae:.2f}")

    # checkpoint every epoch, in case of a session interruption (learned this lesson already)
    torch.save(model.state_dict(), "/kaggle/working/hybrid_age_model.pt")

print("Training complete. Final model saved to /kaggle/working/hybrid_age_model.pt")

Epoch 1/10: 100%|██████████| 879/879 [02:14<00:00,  6.54it/s]


Epoch 1: train_loss=1.8642 train_MAE=11.54 val_MAE=9.81


Epoch 2/10: 100%|██████████| 879/879 [02:20<00:00,  6.25it/s]


Epoch 2: train_loss=1.7011 train_MAE=9.74 val_MAE=9.36


Epoch 3/10: 100%|██████████| 879/879 [02:19<00:00,  6.28it/s]


...

Epoch 8: train_loss=1.4202 train_MAE=7.34 val_MAE=8.71


Epoch 9/10: 100%|██████████| 879/879 [02:21<00:00,  6.22it/s]


Epoch 9: train_loss=1.3836 train_MAE=7.10 val_MAE=8.44


Epoch 10/10: 100%|██████████| 879/879 [02:20<00:00,  6.23it/s]


Epoch 10: train_loss=1.3465 train_MAE=6.81 val_MAE=8.67
Training complete. Final model saved to /kaggle/working/hybrid_age_model.pt


In [8]:
import numpy as np

In [9]:
model.eval()

def predict_with_confidence(model, imgs):
    with torch.no_grad():
        logits = model(imgs.to(device))
        probs = F.softmax(logits, dim=1)
        classes = torch.arange(NUM_CLASSES, device=device).float()
        pred_class = (probs * classes).sum(dim=1)
        variance = (probs * (classes - pred_class.unsqueeze(1)) ** 2).sum(dim=1)
        std_dev = torch.sqrt(variance)
    return pred_class.cpu(), std_dev.cpu()


all_preds, all_true, all_std = [], [], []

for imgs, ages in tqdm(test_loader, desc="Evaluating on test set"):
    preds, stds = predict_with_confidence(model, imgs)
    all_preds.extend((preds + MIN_AGE).tolist())   # convert back to real age
    all_true.extend((ages + MIN_AGE).tolist())      # convert back to real age
    all_std.extend(stds.tolist())

all_preds = np.array(all_preds)
all_true = np.array(all_true)
all_std = np.array(all_std)

test_mae = np.mean(np.abs(all_preds - all_true))
print(f"Test set MAE: {test_mae:.2f} years")

Evaluating on test set: 100%|██████████| 189/189 [00:53<00:00,  3.51it/s]

Test set MAE: 8.77 years


In [10]:
def threshold_accuracy(preds, trues, threshold):
    true_side = trues >= threshold
    pred_side = preds >= threshold
    return (true_side == pred_side).mean()

for t in [13, 18, 21]:
    acc = threshold_accuracy(all_preds, all_true, t)
    print(f"Threshold accuracy @ age {t}: {acc:.1%}")

Threshold accuracy @ age 13: 96.7%
Threshold accuracy @ age 18: 91.7%
Threshold accuracy @ age 21: 88.4%


In [11]:
from scipy.stats import norm

def check_calibration(preds, trues, stds, confidence_level=0.90):
    z = norm.ppf(0.5 + confidence_level / 2)
    lower = preds - z * stds
    upper = preds + z * stds
    covered = ((trues >= lower) & (trues <= upper)).mean()
    print(f"Target coverage: {confidence_level:.0%}, Empirical coverage: {covered:.1%}")
    if abs(covered - confidence_level) < 0.05:
        print("Well-calibrated.")
    elif covered < confidence_level:
        print("Overconfident — ranges are too narrow. Consider widening std by a correction factor.")
    else:
        print("Underconfident — ranges are wider than necessary.")
    return covered

check_calibration(all_preds, all_true, all_std)

Target coverage: 90%, Empirical coverage: 83.6%
Overconfident — ranges are too narrow. Consider widening std by a correction factor.


np.float64(0.8356300846754109)

In [12]:
!pip install -q --no-deps onnxscript onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 10.1 MB/s eta 0:00:00


In [13]:
model.eval()
dummy_input = torch.randn(1, 3, 224, 224, device=device)

torch.onnx.export(
    model,
    dummy_input,
    "/kaggle/working/hybrid_age_model.onnx",
    input_names=["image"],
    output_names=["logits"],
    dynamic_axes={"image": {0: "batch_size"}, "logits": {0: "batch_size"}},
    opset_version=13,
    dynamo=False,  # forces the legacy TorchScript-based exporter, avoids the onnxscript dependency
)

import os
size_mb = os.path.getsize("/kaggle/working/hybrid_age_model.onnx") / (1024*1024)
print(f"Exported ONNX model: {size_mb:.1f} MB")

/tmp/ipykernel_23/1342180784.py:4: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/torch/__init__.py:2228: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert condition, message


Exported ONNX model: 202.9 MB


In [14]:
!pip install -q --no-deps onnxruntime

import onnxruntime as ort

session = ort.InferenceSession("/kaggle/working/hybrid_age_model.onnx")

# grab one real test batch and compare PyTorch vs ONNX outputs
sample_imgs, sample_ages = next(iter(test_loader))

with torch.no_grad():
    pytorch_logits = model(sample_imgs.to(device)).cpu().numpy()

onnx_logits = session.run(None, {"image": sample_imgs.numpy()})[0]

max_diff = np.abs(pytorch_logits - onnx_logits).max()
print(f"Max difference between PyTorch and ONNX outputs: {max_diff:.6f}")
print("Match:" , "YES - safe to deploy" if max_diff < 1e-3 else "NO - investigate before deploying")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 68.5 MB/s eta 0:00:00
Max difference between PyTorch and ONNX outputs: 0.000044
Match: YES - safe to deploy
